# Modelos jerárquicos

¿Cómo podemos aprender sobre varios grupos sin suponer que son todos iguales ni analizarlos por separado?

Vamos a trabajar con dos problemas: primero, estudiaremos la supervivencia durante operaciones cardíacas en diferentes hospitales, luego analizaremos puntajes de matemática en  diferentes escuelas.

In [37]:
# Instalación opcional en Google Colab
import sys

if "google.colab" in sys.modules:
    %pip install -q "pymc==6.4.0" "preliz==0.28.0" "arviz-base==1.3.1" "matplotlib==3.11.2" "nutpie==0.16.11"
    %pip install -q polars h5netcdf h5py graphviz

In [38]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import preliz as pz
import pymc as pm
import xarray as xr

from time import perf_counter

COLORES = ["#0072B2", "#E69F00", "#009E73", "#CC79A7"]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=COLORES)
az.rcParams["stats.ci_kind"] = "eti"
az.rcParams["stats.ci_prob"] = 0.95

rng = np.random.default_rng(2026)

## Caso 1: Supervivencia en hospitales

En 12 hospitales se realizaron operaciones cardíacas a bebés.
Tenemos el número de operaciones ($n_i$) realizadas y la cantidad de bebés que sobrevivieron en cada hospital ($y_i$).
Representamos a la probabilidad de supervivencia en el hospital $i$ con $\pi_i$.

Vamos a construir un modelo basado en la distribución binomial donde el éxito es sobrevivir a la operación. Nos interesa responder:

- **¿Cuál es la probabilidad de supervivencia en cada hospital y con cuánta precisión podemos estimarla?**

In [39]:
df_hospitales = pl.read_csv("https://drive.google.com/uc?export=download&id=12BUWiFqe3v5oxh0r0tPirHtUQdxT1hib")

with pl.Config(tbl_rows=12):
    display(df_hospitales)

hospital,operaciones,supervivientes,proporcion
str,i64,i64,f64
"""A""",5,5,1.0
"""B""",15,13,0.866667
"""C""",12,11,0.916667
"""D""",81,76,0.938272
"""E""",21,20,0.952381
"""F""",20,18,0.9
"""G""",15,14,0.933333
"""H""",22,18,0.818182
"""I""",21,19,0.904762


### Modelo 1: hospitales independientes

Ya vimos cómo representar la incertidumbre sobre una probabilidad con un modelo beta-binomial.
Podemos ajustar uno por hospital, sin que los resultados de los demás intervengan en la actualización de su información:

$$
\begin{aligned}
Y_i\mid \pi_i&\sim\operatorname{Binomial}(n_i,\pi_i)\\
\pi_i&\overset{\mathrm{ind}}\sim\operatorname{Beta}(1,1)
\end{aligned}
$$

con $i = 1, \dots, 12$.

A este modelo se lo conoce como modelo **sin agrupamiento**, o _no pooling_, y lo implementamos en PyMC:

In [ ]:
# Creamos variables con cantidades necesarias...
n_operaciones = df_hospitales["operaciones"].to_numpy()
supervivientes = df_hospitales["supervivientes"].to_numpy()

hospitales = df_hospitales["hospital"].to_list()
coords_hospital = {"hospital": hospitales}

with pm.Model(coords=coords_hospital) as modelo_h_ind:
    pi = pm.Beta("pi", alpha=1, beta=1, dims="hospital")
    pm.Binomial("y", n=n_operaciones, p=pi, observed=supervivientes, dims="hospital")

modelo_h_ind.to_graphviz()

Lo ajustamos:

In [ ]:
with modelo_h_ind:
    idata_h_ind = pm.sample(random_seed=1, chains=4)

Y exploramos los diagnósticos del muestreo:

In [ ]:
az.summary(idata_h_ind, var_names=["pi"], kind="diagnostics")

Dado que los diagnósticos no presentan ningún problema, visualizamos un resumen de la distribución _a posteriori_ de $\pi$ para cada hospital.

En ese caso usamos `az.plot_forest` que genera un gráfico con la media e intervalos de credibilidad del 50% y del 95%.

In [ ]:
az.plot_forest(idata_h_ind, var_names="pi", combined=True, figure_kwargs={"figsize": (7, 4)});

Si queremos visualizar directamente las distribuciones, podemos usar `az.plot_dist`.
La opción `sharex` igual a `True` pone a todos los paneles en una misma escala para el eje horizontal para facilitar la compración.

In [ ]:
az.plot_dist(
    idata_h_ind,
    var_names="pi",
    figure_kwargs={
        "layout": "constrained",
        "sharex": True,
        "figsize": (14, 6)
    }
);

Ahora tenemos una distribución para la probabilidad de supervivencia en cada hospital.
En el hospital A sobrevivieron los 5 bebés operados y la proporción observada es 1.

Eso no implica que la supervivencia esté garantizada en ese hospital.
De hecho, la distribución _a posteriori_ permite representar esa incertidumbre,
en lugar de quedarnos con una estimación puntual igual a 1.

Pero esta estimación no usa nada sobre lo que ocurrió en los otros once hospitales. ¿Podría servirnos esa información?

### Modelo 2: hospitales idénticos

Otra posibilidad es suponer que hay una probabilidad de éxito común para todos los hospitales $\pi$.

Esto permite juntar las operaciones y supervivientes de todos los hospitales en un único registro.
El modelo resulta:

$$
\begin{aligned}
\sum_i Y_i\mid \pi&\sim\operatorname{Binomial}\left(\sum_i n_i,\pi\right),\\
\pi&\sim\operatorname{Beta}(1,1).
\end{aligned}
$$

A este modelo se lo conoce como de **agrupamiento completo**, o modelo _complete pooling_.

In [ ]:
n_total = sum(n_operaciones)
y_total = sum(supervivientes)

with pm.Model() as modelo_h_completo:
    pi = pm.Beta("pi", alpha=1, beta=1)
    pm.Binomial("y", n=n_total, p=pi, observed=y_total)

modelo_h_completo.to_graphviz()

In [ ]:
with modelo_h_completo:
    idata_h_completo = pm.sample(random_seed=2, chains=4)

Si no pasamos ningun `kind` a `az.summary`, obtenemos tanto los resúmenes estadísticos como los diagnósticos para la inferencia mediante MCMC.

In [ ]:
az.summary(idata_h_completo, var_names=["pi"])

Ahora, podemos comparar la distribución _a posteriori_ obtenida con el modelo sin agrupamiento (_no pooling_) y el modelo de agrupamiento completo (_complete pooling_)

In [ ]:
pc = az.plot_forest(
    {
        "No pooling": idata_h_ind,
        "Complete pooling": idata_h_completo,
    },
    var_names="pi",
    combined=True,
    figure_kwargs={"figsize": (7, 4)}
);
# pc.add_legend("model");

La probabilidad común se estima con todas las operaciones y ahora obtenemos una distribución _a posteriori_ con menor incertidumbre.

Sin embargo, la respuesta para cualquier hospital ahora es la misma.

Ganamos precisión para hablar sobre la probabilidad de supervivencia en una operación en cualquier hospital, pero perdimos la capacidad de discriminar entre ellos.

Si queremos aprender sobre las diferencias entre hospitales, este supuesto nos queda corto.

### Modelo 3: hospitales relacionados

Podemos conservar una probabilidad de supervivencia $\pi_i$ por hospital y vincularlas mediante una distribución común:

$$
\pi_i\mid\alpha,\beta\sim\operatorname{Beta}(\alpha,\beta).
$$

Los parámetros $\alpha$ y $\beta$ describen esa distribución y se llaman **hiperparámetros**. Como desconocemos sus valores, también les asignamos una distribución *a priori*, $p(\alpha,\beta)$.

Con los datos obtendremos una distribución _a posteriori_ conjunta para las probabilidades de los hospitales y los hiperparámetros.
Cada hospital aporta información sobre $\alpha$ y $\beta$, y esas cantidades compartidas intervienen, a su vez, en la estimación de todos los $\pi_i$.

Este mecanismo se conoce como **agrupamiento parcial**, o _partial pooling_.

Esta construcción trata a todos los hospitales de la misma manera antes de observar los datos, ninguno recibe un prior distinto por ser quien es.

El supuesto se llama **intercambiabilidad** (*exchangeability*). Formalmente, cambiar las etiquetas de los hospitales no modifica el prior conjunto sobre las probabilidades de éxito de los hospitales:

$$
p(\pi_1,\ldots,\pi_{12}) = p(\pi_{\rho(1)},\ldots,\pi_{\rho(12)})
$$

para cualquier permutación $\rho$.

Dados $\alpha$ y $\beta$, se supone que las probabilidades son independientes.
Al desconocer esos hiperparámetros, las probabilidades $\pi_i$ quedan vinculadas mediante su incertidumbre compartida.

Comparemos ahora las tres formas de organizar el modelo mediante diagramas. Los parámetros individuales están en azul, los hiperparámetros compartidos en naranja y los conteos observados en verde. Omitimos los tamaños conocidos $n_i$ para simplificar los dibujos.

#### _No pooling_

<img src="https://drive.google.com/uc?export=view&id=1z9_TVEeRIu0VrphZuOjr71nNgIE705E1" width="600px">

Con priors independientes de parámetros fijos, la distribución _a posteriori_ se separa por hospital:

$$
p(\boldsymbol\pi\mid\mathbf y,\mathbf n)
\propto\prod_{i=1}^{12}p(y_i\mid n_i,\pi_i)\,p(\pi_i)
\quad\Longrightarrow\quad
p(\boldsymbol\pi\mid\mathbf y,\mathbf n)
=\prod_{i=1}^{12}p(\pi_i\mid y_i,n_i).
$$

Los datos de un hospital actualizan únicamente su propia probabilidad de éxito.

#### _Complete pooling_

<img src="https://drive.google.com/uc?export=view&id=1SFDUQDPlcBxDYpsA5HerajCzz4ePHzim" width="600px">

Se agrupan los datos de todos los hospitales y hay un solo parámetro que se actualiza con los datos de todos los hospitales:

$$
p(\pi\mid\mathbf y,\mathbf n)
\propto p(\pi)\prod_{i=1}^{12}p(y_i\mid n_i,\pi).
$$

Cada hospital contribuye a aprender la misma probabilidad y el modelo no permite diferencias individuales.

#### _Partial pooling_

<img src="https://drive.google.com/uc?export=view&id=104xeE9bDPmWy4wRz7PR9a5BlAgFpmHFb" width="600px">


La distribución _a posteriori_ conjunta incluye las probabilidades de los hospitales y los hiperparámetros:

$$
p(\boldsymbol\pi,\alpha,\beta\mid\mathbf y,\mathbf n)
\propto p(\alpha,\beta)
\prod_{i=1}^{12}p(y_i\mid n_i,\pi_i)\,p(\pi_i\mid\alpha,\beta).
$$

Con los hiperpriors independientes que especificamos a continuación, $p(\alpha,\beta)=p(\alpha)p(\beta)$.

Las probabilidades de los hospitales provienen de una distribución beta común, cuyos parámetros
$\alpha$ y $\beta$ también son desconocidos:

$$
\begin{aligned}
Y_i\mid\pi_i&\sim\operatorname{Binomial}(n_i,\pi_i),\\
\pi_i\mid\alpha,\beta&\sim\operatorname{Beta}(\alpha,\beta),\\
\alpha&\sim\operatorname{Gamma}(3, 2),\\
\beta&\sim\operatorname{Gamma}(3, 2).
\end{aligned}
$$

Usamos la parametrización forma–tasa para las gamma y priors independientes para $\alpha$ y $\beta$.
Sus medias y varianza son 1.5 y 0.75, respectivamente, permiten incertidumbre sobre la forma de la beta común.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3), layout="constrained")

alpha_rv = pz.Gamma(alpha=3, beta=2)
beta_rv = pz.Gamma(alpha=3, beta=2)

alpha_rv.plot_pdf(ax=axes[0], legend=False)
beta_rv.plot_pdf(ax=axes[1], legend=False)

axes[0].set(title="Prior de α", xlabel="α")
axes[1].set(title="Prior de β", xlabel="β");

Extrayendo musestras del _prior_ de $\alpha$ y $\beta$ podemos visualizar el _prior_ implícito sobre $\pi_i$.

In [ ]:
alpha_rvs = alpha_rv.rvs(10_000, random_state=1)
beta_rvs = beta_rv.rvs(10_000, random_state=1)
pi_rvs = pz.Beta(alpha_rvs, beta_rvs).rvs(random_state=1)

fig, ax = plt.subplots()
ax.hist(pi_rvs, bins=40, density=True)
ax.set(xlabel="$\\pi$", ylabel="$p(\\pi)$", title="Prior de $\\pi$");

Ahora, implementemos el modelo en PyMC:

In [ ]:
with pm.Model(coords=coords_hospital) as modelo_h:
    alpha = pm.Gamma("alpha", alpha=3, beta=2)
    beta = pm.Gamma("beta", alpha=3, beta=2)
    pi = pm.Beta("pi", alpha=alpha, beta=beta, dims="hospital")
    pm.Binomial("y", n=n_operaciones, p=pi, observed=supervivientes, dims="hospital")

modelo_h.to_graphviz()

In [ ]:
with modelo_h:
    idata_h = pm.sample(random_seed=3, chains=4)

In [ ]:
az.summary(idata_h, kind="diagnostics")

In [ ]:
az.plot_dist(idata_h, var_names=["alpha", "beta"]);

In [ ]:
az.plot_dist(
    idata_h,
    var_names="pi",
    figure_kwargs={
        "layout": "constrained",
        "sharex": True,
        "figsize": (14, 6)
    }
);

### Comparación de $\pi$ según modelo

Comparemos las estimaciones sin agrupamiento y con agrupamiento parcial sobre los mismos datos. En cada caso vamos a mostrar la media y un intervalo central del 95%, junto con las proporciones observadas.

In [ ]:
pi_observada = supervivientes / n_operaciones
pi_ind = idata_h_ind["posterior"]["pi"].to_numpy().reshape(-1, 12)
pi_parcial = idata_h["posterior"]["pi"].to_numpy().reshape(-1, 12)

fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
for muestras, desplazamiento, etiqueta, color in [
    (pi_ind, -0.12, "Sin pooling", "C0"),
    (pi_parcial, 0.12, "Pooling parcial", "C1"),
]:
    media = muestras.mean(axis=0)
    inferior, superior = np.quantile(muestras, [0.025, 0.975], axis=0)
    ax.errorbar(
        np.arange(12) + desplazamiento,
        media,
        yerr=[media - inferior, superior - media],
        fmt="o",
        color=color,
        label=etiqueta,
    )
ax.scatter(np.arange(12), pi_observada, marker="x", color="0.3", label="Proporción observada")
ax.axhline(1, ls="--", color="0.5")
ax.set_ylim(0.5, 1.05)

for i, (y_i, n_i) in enumerate(zip(supervivientes, n_operaciones)):
    ax.text(
        i,
        1.02,
        f"{y_i}/{n_i}",
        ha="center",
        fontsize=9,
        color="0.25"
    )
ax.set(
    xticks=np.arange(12),
    xticklabels=hospitales,
    xlabel="Hospital",
    ylabel="Probabilidad de supervivencia",
    title="Supervivientes / operaciones por hospital"
)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False);

Las medias _a posteriori_ de las probabilidades de supervivencia quedan más cerca unas de otras que las proporciones observadas. Este efecto se conoce como **contracción** (*shrinkage*). El cambio suele ser mayor en los hospitales con menos operaciones, porque sus datos aportan menos información para estimar la probabilidad de supervivencia.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

hospitals_sort_idx = np.argsort(n_operaciones)
hospitals = np.array(list("ABCDEFGHIJKL"))
pi_diff = pi_parcial.mean(0) - pi_observada

ax.scatter(hospitals[hospitals_sort_idx], pi_diff[hospitals_sort_idx])
ax.axhline(0, color="0.5", ls="--")

for i, n_i in enumerate(n_operaciones[hospitals_sort_idx]):
    ax.text(i, 0.005, f"n={n_i}", ha="center", fontsize=9, color="0.2")

ax.set(ylim=(-0.1, 0.01));

## Caso 2: Puntajes de matemática en escuelas

Tenemos los puntajes de una prueba de matemática de 1.993 estudiantes de décimo grado, distribuidos en 100 escuelas de Estados Unidos. Los datos provienen del estudio ELS de 2002.

Las escuelas son secundarias públicas urbanas con al menos 400 estudiantes matriculados en décimo grado. De cada una observamos entre 4 y 32 estudiantes: la cantidad de puntajes en la muestra es distinta de la matrícula de la escuela. Cada fila del archivo corresponde a un estudiante.

Los puntajes van de 0 a 100 y están estandarizados para que la media nacional sea 50 y el desvío sea 10. La media de las escuelas que vamos a estudiar puede diferir de ese valor.

Con estos datos queremos responder:

- **¿Cuál es el puntaje medio de cada escuela y con qué incertidumbre podemos estimarlo?**
- **¿Cuánto difieren las medias entre escuelas y cómo influye el tamaño de muestra en las estimaciones?**

In [ ]:
df_escuelas = pl.read_csv("https://drive.google.com/uc?export=download&id=1XbbpcJYgK8adOEZ8CM-xv_VBZ-z4Wu44")
df_escuelas

Para visualizar los puntajes obtenidos elegimos 15 escuelas al azar.
Dibujamos un boxplot por escuela y agregamos los puntajes de sus estudiantes,
con un pequeño desplazamiento horizontal para que se puedan distinguir.

In [ ]:
rng_escuelas = np.random.default_rng(95)
escuelas_elegidas = np.sort(
    rng_escuelas.choice(df_escuelas["school"].unique().sort().to_numpy(), size=15, replace=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
for i, escuela in enumerate(escuelas_elegidas):
    puntajes_escuela = df_escuelas.filter(pl.col("school") == escuela)["mathscore"].to_numpy()
    jitter = rng_escuelas.normal(0, 0.05, size=len(puntajes_escuela))
    ax.boxplot(puntajes_escuela, positions=[i], widths=0.4, showfliers=False)
    ax.scatter(i + jitter, puntajes_escuela, alpha=0.6, s=20)

ax.set(
    xlabel="Escuela", ylabel="Puntaje de matemática",
    xticks=np.arange(len(escuelas_elegidas)), xticklabels=escuelas_elegidas,
    ylim=(0, 100),
);

### Modelo jerárquico normal

Ahora $i$ identifica una escuela y $j$ un estudiante. Usamos $Y_{ij}$ para el puntaje como variable aleatoria y $y_{ij}$ para el puntaje observado.

Le damos a cada escuela una media $\mu_i$ y suponemos que esas medias tienen una distribución normal común:

$$
\begin{aligned}
Y_{ij}\mid\mu_i,\sigma&\sim\operatorname{Normal}(\mu_i,\sigma^2),\\
\mu_i\mid\mu_0,\tau&\sim\operatorname{Normal}(\mu_0,\tau^2).
\end{aligned}
$$

$\mu_0$ es la media de la población de medias escolares y $\tau$, su desvío.

Por otro lado, el desvío $\sigma$ describe la variación entre estudiantes de una misma escuela y es común a todas las escuelas.

### Elucidación de _priors_

Los puntajes van de 0 a 100 y el examen tiene media nacional 50 y desvío 10. Para especificar los priors,
vamos a expresar lo que esperamos mediante intervalos:

- Asignamos un 95% de probabilidad a que la media poblacional esté entre 30 y 70 puntos.
- Para el desvío entre estudiantes de una misma escuela, asignamos un 95% al intervalo de 1 a 20 puntos.
- Para el desvío entre las medias escolares, asignamos un 95% al intervalo de 1 a 10 puntos.

Usamos una normal para la media poblacional e inversas gamma para las varianzas. Como los dos últimos
intervalos se refieren a desvíos, elevamos sus extremos al cuadrado al trabajar con las varianzas.
`pz.maxent` busca los parámetros de mayor entropía dentro de cada familia bajo esas restricciones.
Estas son elecciones previas al ajuste; no las calculamos con las medias ni los desvíos de esta muestra.

In [ ]:
prior_mu = pz.maxent(pz.Normal(mu=50), lower=30, upper=70, mass=0.95, plot=False)
prior_sigma2 = pz.maxent(pz.InverseGamma(), lower=1**2, upper=20**2, mass=0.95, plot=False)
prior_tau2 = pz.maxent(pz.InverseGamma(), lower=1**2, upper=10**2, mass=0.95, plot=False)

print(prior_mu)
print(prior_sigma2)
print(prior_tau2)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), layout="constrained")

prior_mu.plot_pdf(ax=axes[0], legend=False)
prior_sigma2.plot_pdf(ax=axes[1], legend=False)
prior_tau2.plot_pdf(ax=axes[2], legend=False)

axes[0].set(xlabel=r"$\mu_0$")
axes[1].set(xlabel=r"$\sigma^2$")
axes[2].set(xlabel=r"$\tau^2$");

### El modelo jerárquico

Completamos la especificación con los priors que acabamos de obtener.

<img src="https://drive.google.com/uc?export=view&id=1eAh2kLd0A_HLOVt19sn05uqKVxyO2SSb" width="600px">


El modelo completo es:

$$
\begin{aligned}
Y_{ij}\mid\mu_i,\sigma&\sim\operatorname{Normal}(\mu_i,\sigma^2),\\
\mu_i\mid\mu_0,\tau&\sim\operatorname{Normal}(\mu_0,\tau^2),\\
\mu_0&\sim\operatorname{Normal}(50,10.20^2),\\
\sigma^2&\sim\operatorname{InvGamma}(6.232,622.555),\\
\tau^2&\sim\operatorname{InvGamma}(4.728,180.177).
\end{aligned}
$$

$\sigma$ mide la variación entre estudiantes de la misma escuela y $\tau$, entre las medias de escuelas.


En PyMC:

In [ ]:
escuelas, escuela_idx = np.unique(df_escuelas["school"].to_numpy(), return_inverse=True)
puntajes = df_escuelas["mathscore"].to_numpy()

In [ ]:
coords_escuelas = {
    "escuela": escuelas,
    "estudiante": np.arange(len(puntajes)),
}

with pm.Model(coords=coords_escuelas) as modelo_escuelas:
    mu_0 = pm.Normal("mu_0", mu=prior_mu.mu, sigma=prior_mu.sigma)
    tau2 = pm.InverseGamma("tau2", alpha=prior_tau2.alpha, beta=prior_tau2.beta)
    sigma2 = pm.InverseGamma("sigma2", alpha=prior_sigma2.alpha, beta=prior_sigma2.beta)

    mu_escuela = pm.Normal("mu_escuela", mu=mu_0, sigma=tau2**0.5, dims="escuela")

    pm.Normal(
        "puntaje",
        mu=mu_escuela[escuela_idx],
        sigma=sigma2**0.5,
        observed=puntajes,
        dims="estudiante"
    )

modelo_escuelas.to_graphviz()

In [ ]:
with modelo_escuelas:
    idata_escuelas = pm.sample(chains=4, random_seed=2026)

In [ ]:
az.summary(idata_escuelas, var_names=["mu_0", "tau2", "sigma2"], round_to=2)

Podemos explorar la distribución _a posteriori_ de la media de la puntuación para algunas escuelas:

In [ ]:
az.plot_forest(
    idata_escuelas,
    var_names="mu_escuela",
    combined=True,
    coords={"escuela": list(range(1, 101, 10))},
    figure_kwargs={"figsize": (7, 4)}
);

Tammbién podemos estudiar cuánta variabilidad se debe a la diferencia entre estudiantes y cuánta a la diferencia entre escuelas.

La variabilidad total de los puntajes se puede escribir como $\tau^2+\sigma^2$.
La proporción que corresponde a diferencias entre las medias de las escuelas es:

$$
\gamma=\frac{\tau^2}{\tau^2+\sigma^2}.
$$

En este modelo, $\gamma$ también es la correlación entre los puntajes de dos estudiantes de una misma
escuela, dados los parámetros poblacionales. Por eso se conoce como **correlación intraclase** (ICC).

In [ ]:
tau2_draws = idata_escuelas["posterior"]["tau2"]
sigma2_draws = idata_escuelas["posterior"]["sigma2"]
gamma_draws = (tau2_draws / (tau2_draws + sigma2_draws)).to_numpy().flatten()

fig, ax = plt.subplots()

ax.hist(gamma_draws, bins=30, density=True);
ax.set(xlabel=r"$\gamma = \tau^2/(\tau^2+\sigma^2)$", ylabel="Densidad");

In [ ]:
gamma_draws.mean()

In [ ]:
np.quantile(gamma_draws, (0.025, 0.975))

La media _a posteriori_ de $\gamma$ es de aproximadamente 0.23, con un intervalo del 95% de 0.18 a 0.29.

En este modelo, la mayor parte de la variabilidad de los puntajes corresponde a diferencias entre estudiantes dentro de una misma escuela.

### Medias empíricas vs medias _a posteriori_

Comparemos la media observada $\bar y_i$ de cada escuela con la media _a posteriori_ de $\mu_i$.
Si conociéramos $\mu_0$, $\tau$ y $\sigma$, esta última sería un promedio ponderado de la media observada y la media poblacional:

$$
E[\mu_i\mid\mathbf y,\mu_0,\tau,\sigma]
=w_i\bar y_i+(1-w_i)\mu_0,\qquad
w_i=\frac{n_i\tau^2}{n_i\tau^2+\sigma^2}.
$$

El peso $w_i$ aumenta con la cantidad de estudiantes $n_i$. Una escuela con muchos estudiantes aporta más información sobre su propia media. Con pocos estudiantes, la distribución común tiene más peso en la estimación.

En nuestro modelo, $\mu_0$, $\tau$ y $\sigma$ también son desconocidos. Para obtener la media _a posteriori_ de cada escuela, promediamos la expresión anterior sobre la distribución _a posteriori_ de esas cantidades.

In [ ]:
resumen_escuelas = (
    df_escuelas.group_by("school")
    .agg(pl.len().alias("n"), pl.col("mathscore").mean().alias("media"))
    .sort("school")
)
n_escuela = resumen_escuelas["n"].to_numpy()
medias_empiricas = resumen_escuelas["media"].to_numpy()

post_esc = idata_escuelas["posterior"]
mu_esc = post_esc["mu_escuela"]
media_esc = mu_esc.mean(dim=("chain", "draw")).to_numpy()
media_poblacional = post_esc["mu_0"].mean().item()
cambio_esc = media_esc - medias_empiricas

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
limites = [medias_empiricas.min() - 1, medias_empiricas.max() + 1]

ax1.scatter(medias_empiricas, media_esc)
ax1.plot(limites, limites, color="0.5", ls="--")
ax1.axhline(media_poblacional, color="C1", ls=":", label="Media poblacional a posteriori")
ax1.set(xlabel="Media observada", ylabel="Media a posteriori", xlim=limites, ylim=limites)
ax1.legend(fontsize=10)

ax2.scatter(n_escuela, cambio_esc)
ax2.axhline(0, color="0.5", ls="--")
ax2.set(xlabel="Estudiantes observados por escuela", ylabel="Media a posteriori - media observada");

La diagonal del primer panel marca dónde coincidirían las dos medias. Las medias observadas más bajas tienden a subir y las más altas, a bajar: se acercan a la media poblacional.

En el segundo panel, `cambio_esc` mide cuánto se movió cada estimación. Los valores positivos indican un aumento y los negativos, una disminución. Los cambios más grandes aparecen en escuelas con pocos estudiantes, aunque también dependen de cuánto se aleja su media observada de la media poblacional.

Para ver la incertidumbre sobre la media de cada escuela, agreguemos los intervalos centrales del 95%. Mostramos las 100 escuelas, junto con sus medias observadas.

In [ ]:
ic_esc = mu_esc.quantile([0.025, 0.975], dim=("chain", "draw")).to_numpy()
orden = np.argsort(medias_empiricas)
posiciones = np.arange(len(escuelas))

fig, ax = plt.subplots(figsize=(8, 16))
ax.hlines(posiciones, ic_esc[0, orden], ic_esc[1, orden], color="C0", alpha=0.5)
ax.scatter(media_esc[orden], posiciones, s=15, label="Media a posteriori e IC del 95%")
ax.scatter(medias_empiricas[orden], posiciones, marker="x", s=20, color="C1", label="Media observada")
ax.axvline(media_poblacional, color="0.5", ls="--")
ax.set(xlabel="Puntaje medio", ylabel="Escuela", yticks=posiciones, yticklabels=escuelas[orden])
ax.invert_yaxis()
ax.tick_params(axis="y", labelsize=9)
ax.legend();

## Siguientes pasos

En esta clase nos enfocamos en entender cómo se construyen los modelos jerárquicos y cómo comparten información entre grupos.

Estudiamos la contracción (*shrinkage*) y vimos cómo puede ayudarnos a reducir la incertidumbre de las estimaciones, especialmente cuando contamos con pocos registros por grupo.
También vimos cómo las estimaciones más atípicas pueden acercarse a la media poblacional, aunque el tamaño del cambio depende de cuánta información aporta cada grupo.

Para completar el análisis, podríamos realizar **chequeos predictivos a priori**: generar datos usando los priors y los hiperpriors, antes de ajustar el modelo, y revisar si producen valores y patrones plausibles para el problema.

Después del ajuste, los **chequeos predictivos a posteriori** nos permitirían generar nuevos conjuntos de datos y compararlos con los observados. Por ejemplo, podríamos revisar si el modelo reproduce las proporciones de supervivencia de los hospitales o la dispersión de los puntajes dentro de las escuelas y entre ellas.

Los mismos principios pueden aplicarse a modelos más complejos. En una regresión lineal jerárquica, por ejemplo, cada grupo podría tener su propio intercepto y su propia pendiente, vinculados mediante distribuciones comunes. También podemos construir modelos lineales generalizados jerárquicos, como regresiones logísticas para respuestas binarias o modelos de Poisson para conteos.